# 13b — Encoding B personality dual-channel RBM training (5k)

Masked CD-1 training on notebook **12b** tensors:

- **RBMB1:** user-rating one-hot (`channelB1`, visible = `n_movies × K`)
- **RBMB2:** personality scalar (`channelB2`, visible = `n_movies`)

Same CD-1 pattern as notebook **13**, with:
- visible size inferred from `X.shape[1]`
- negative-phase reconstruction masked by `M`
- `BATCH_SIZE=500`, artifacts saved with `_5k` suffix
- **No per-epoch ΔW logging** — only final `W`, `b_h`, and `mse_log`

| Part | Section |
|------|--------|
| Part 0 | Setup |
| Part 1 | Load / flatten / masks |
| Part 2 | Masked CD-1 training |
| Part 3 | Save `_5k` artifacts |
| Part 4 | Verification |

**Prerequisites:** notebooks 09b, 12b.


## Part 0 — Setup

`X_DTYPE`: prefer `float32` (X1 ≈ 2.6 GB); `float64` ≈ 5.3 GB RAM.


In [ ]:
from pathlib import Path
import shutil
import time

import numpy as np

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

proc = root / "data" / "processed"
assert proc.exists(), f"Missing {proc}"

N_HIDDEN = 128
LR = 0.01
BATCH_SIZE = 500
EPOCHS = 50
INIT_SEED = 42
K = 10
X_DTYPE = np.float32        # np.float64 if RAM allows

path_b1 = proc / "channelB1_softmax.npy"
path_b2 = proc / "channelB2_personality.npy"
path_mask = proc / "mask.npy"
for p in (path_b1, path_b2, path_mask):
    assert p.exists(), f"Missing {p} — run notebooks 09b / 12b first."

ch1_meta = np.load(path_b1, mmap_mode="r")
ch2_meta = np.load(path_b2, mmap_mode="r")
mask_meta = np.load(path_mask, mmap_mode="r")
n_users, n_movies, k_b1 = ch1_meta.shape
assert k_b1 == K
assert ch2_meta.shape == (n_users, n_movies, 1)
assert mask_meta.shape == (n_users, n_movies)

n_vis_b1 = n_movies * K
n_vis_b2 = n_movies
bytes_x1 = n_users * n_vis_b1 * np.dtype(X_DTYPE).itemsize
bytes_x2 = n_users * n_vis_b2 * np.dtype(X_DTYPE).itemsize
bytes_w1 = n_vis_b1 * N_HIDDEN * 8
free = shutil.disk_usage(proc).free

print(f"Project root: {root}")
print(f"Cohort: n_users={n_users:,}, n_movies={n_movies:,}, K={K}")
print(f"\n=== Memory / disk estimate ===")
print(f"  X1 ({X_DTYPE.__name__}):     {bytes_x1 / 1e9:.2f} GB   shape=({n_users}, {n_vis_b1})")
print(f"  X2 ({X_DTYPE.__name__}):     {bytes_x2 / 1e9:.2f} GB   shape=({n_users}, {n_vis_b2})")
print(f"  W1 (float64, RAM): {bytes_w1 / 1e6:.1f} MB")
print(f"  Disk free:         {free / 1e9:.2f} GB")
print(f"\nHyperparams: N_HIDDEN={N_HIDDEN}, LR={LR}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, SEED={INIT_SEED}")
print("No deltaW logging — only final W / b_h / mse_log will be saved.")
print("Proceed to Part 1.")

del ch1_meta, ch2_meta, mask_meta


## Part 1 — Load data / flatten / masks

In [ ]:
print("Loading channelB1 (may take a minute) …")
channelB1 = np.load(path_b1)  # float32 dense
print("Loading channelB2 …")
channelB2 = np.load(path_b2)
mask = np.load(path_mask)

n_users, n_movies, _ = channelB1.shape
assert channelB2.shape == (n_users, n_movies, 1)
assert mask.shape == (n_users, n_movies)

X1 = channelB1.reshape(n_users, -1)
X2 = channelB2.reshape(n_users, -1)
if X_DTYPE != np.float32:
    X1 = X1.astype(X_DTYPE)
    X2 = X2.astype(X_DTYPE)
    del channelB1, channelB2  # free float32 originals after cast
else:
    # X1/X2 are views into channel*; keep refs alive via X*
    del channelB1, channelB2

M1 = np.repeat(mask.astype(X_DTYPE, copy=False), K, axis=1)  # (n_users, n_movies*K)
M2 = mask.astype(X_DTYPE, copy=False)                         # (n_users, n_movies)

print(f"X1 {X1.shape} {X1.dtype}")
print(f"X2 {X2.shape} {X2.dtype}")
print(f"M1 {M1.shape} {M1.dtype}")
print(f"M2 {M2.shape} {M2.dtype}")

nz2 = M2.sum(axis=1)
nz1 = M1.sum(axis=1)
assert np.allclose(nz1, K * nz2), "M1 nonzero count must be K × M2 per user"
print(f"✓ M1 / M2 nonzero ratio = K={K} for all users")
print(f"  train movies/user: min={int(nz2.min())}, median={float(np.median(nz2)):.0f}, max={int(nz2.max())}")


## Part 2 — Masked CD-1 training

After negative-phase `v_recon_prob` / `v_recon`, multiply by the batch mask so test/unrated visibles contribute no gradient.

No per-epoch ΔW tensor is kept — only final weights and MSE at epochs 1, 25, 50.


In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60, 60)))


def reconstruction_mse(X, M, W, b_h):
    """Masked MSE: only positions where M==1."""
    h_prob = sigmoid(X @ W + b_h)
    v_prob = sigmoid(h_prob @ W.T)
    diff2 = (X - v_prob) ** 2
    denom = float(M.sum())
    return float((diff2 * M).sum() / denom) if denom > 0 else float("nan")


def train_rbm(X, M, n_hidden, seed, channel_name):
    """Masked CD-1; return W, b_h, mse_log, elapsed_sec."""
    rng = np.random.default_rng(seed)
    n_samples, n_visible = X.shape
    assert M.shape == X.shape, f"Mask shape {M.shape} != X shape {X.shape}"

    W = rng.normal(0.0, 0.01, size=(n_visible, n_hidden)).astype(np.float64)
    b_h = np.zeros(n_hidden, dtype=np.float64)
    mse_log = {}
    t0_all = time.perf_counter()

    print(f"\n=== Training {channel_name} ===")
    print(f"  samples={n_samples}, visible={n_visible}, hidden={n_hidden}")
    print(f"  lr={LR}, batch={BATCH_SIZE}, epochs={EPOCHS}, init_seed={seed}")

    for epoch in range(EPOCHS):
        t0 = time.perf_counter()
        epoch_delta = np.zeros_like(W)
        order = rng.permutation(n_samples)

        for start in range(0, n_samples, BATCH_SIZE):
            idx = order[start : start + BATCH_SIZE]
            v_data = X[idx]
            m_batch = M[idx]
            bs = v_data.shape[0]

            # Positive phase (v_data already zero on test/unrated)
            h_prob = sigmoid(v_data @ W + b_h)
            h_data = (rng.random(h_prob.shape) < h_prob).astype(np.float64)

            # Negative phase + mask enforcement
            v_recon_prob = sigmoid(h_data @ W.T)
            v_recon_prob = v_recon_prob * m_batch
            v_recon = (rng.random(v_recon_prob.shape) < v_recon_prob).astype(np.float64)
            v_recon = v_recon * m_batch

            h_recon_prob = sigmoid(v_recon @ W + b_h)
            h_recon = (rng.random(h_recon_prob.shape) < h_recon_prob).astype(np.float64)

            pos = v_data.T @ h_data
            neg = v_recon.T @ h_recon
            dW = LR * (pos - neg) / bs
            epoch_delta += dW

            db_h = LR * (h_prob.mean(axis=0) - h_recon_prob.mean(axis=0))
            b_h += db_h

        W += epoch_delta

        ep = epoch + 1
        if ep in (1, 25, EPOCHS):
            mse = reconstruction_mse(X, M, W, b_h)
            mse_log[ep] = mse
            print(f"  Epoch {ep:02d} | masked recon MSE: {mse:.6f}")

        if ep == 1:
            dt = time.perf_counter() - t0
            print(f"  ⏱ epoch 1 wall time: {dt:.1f}s  → est. full {EPOCHS} epochs ≈ {dt * EPOCHS / 60:.1f} min")

    elapsed = time.perf_counter() - t0_all
    print(f"  Done in {elapsed / 60:.1f} min")
    return W, b_h, mse_log, elapsed


W1, bh1, mse1, t1 = train_rbm(
    X1, M1, N_HIDDEN, seed=INIT_SEED, channel_name="RBMB1 (user rating)"
)
W2, bh2, mse2, t2 = train_rbm(
    X2, M2, N_HIDDEN, seed=INIT_SEED, channel_name="RBMB2 (personality)"
)
total_train_sec = t1 + t2
print(f"\nTotal training wall time: {total_train_sec / 60:.1f} min")


## Part 3 — Save `_5k` artifacts

In [ ]:
paths = {
    "W1": proc / "rbmB1_weights_5k.npy",
    "W2": proc / "rbmB2_weights_5k.npy",
    "bh1": proc / "rbmB1_bias_hidden_5k.npy",
    "bh2": proc / "rbmB2_bias_hidden_5k.npy",
}

need = W1.nbytes + W2.nbytes + bh1.nbytes + bh2.nbytes
free = shutil.disk_usage(proc).free
print(f"Save check: need ~{need / 1e6:.1f} MB, free {free / 1e9:.2f} GB")
if free < need + 50_000_000:
    raise OSError("Not enough disk to save weight artifacts.")

np.save(paths["W1"], W1)
np.save(paths["W2"], W2)
np.save(paths["bh1"], bh1)
np.save(paths["bh2"], bh2)

meta = {
    "EPOCHS": EPOCHS,
    "N_HIDDEN": N_HIDDEN,
    "BATCH_SIZE": BATCH_SIZE,
    "LR": LR,
    "INIT_SEED": INIT_SEED,
    "mse1": mse1,
    "mse2": mse2,
    "total_train_sec": total_train_sec,
}
np.save(proc / "rbmB_5k_train_meta.npy", meta)

for k, p in paths.items():
    print(f"Saved {k}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")
print(f"Saved meta: {proc / 'rbmB_5k_train_meta.npy'}")


## Part 4 — Verification

In [ ]:
def print_W_stats(name, W):
    print(f"{name}: shape={W.shape}")
    print(
        f"  mean={W.mean():.6f}  std={W.std():.6f}  "
        f"min={W.min():.6f}  max={W.max():.6f}"
    )


print("=== Final weights ===")
print_W_stats("W1 (RBMB1)", W1)
print_W_stats("W2 (RBMB2)", W2)

print("\n=== Masked recon MSE ===")
for ep in (1, 25, EPOCHS):
    print(f"  epoch {ep:2d}:  B1={mse1.get(ep, float('nan')):.6f}  B2={mse2.get(ep, float('nan')):.6f}")

# Coarse channel-structure check: mean |W| per hidden unit
w1_col = np.mean(np.abs(W1), axis=0)  # (n_hidden,)
w2_col = np.mean(np.abs(W2), axis=0)
r = float(np.corrcoef(w1_col, w2_col)[0, 1])
print(f"\nPearson r between mean|W| per hidden unit (B1 vs B2): {r:.4f}")
print("(coarse redundancy check — different visible dims, not a matrix Frobenius correlation)")

print(f"\nTotal training time: {total_train_sec / 60:.1f} min ({total_train_sec:.0f}s)")
print("\nAll done.")
